In [ ]:
from typing import List
from collections import Counter
from verl.utils.reward_score.latex_math import math_is_equivalent, extract_solution

def pass_at_1_from_scores(scores: List[List[int]]) -> float:
    """
    DeepSeek-style pass@1 estimator from k samples per problem.
    """
    total, correct = 0, 0
    for row in scores:
        correct += sum(int(x) for x in row)
        total   += len(row)
    return (correct / total) if total else 0.0

def consensus_at_k(answers, k, gts):
    """Majority-vote accuracy from first K answers per problem."""
    n = len(answers)
    hits = 0
    for preds, gt in zip(answers, gts):
        topk = preds[:k]
        freq = Counter(topk)
        max_freq = max(freq.values())
        winners = sorted([a for a, f in freq.items() if f == max_freq])
        voted = winners[0]
        hits += math_is_equivalent(voted, gt)
    return hits / n if n else 0.0

In [ ]:
TEST_DIRS=[
    "data/test_aime2024/test.parquet",
    "data/test_aime2025/test.parquet",
    "data/test_amc/test.parquet",
    "data/test_brumo2025/test.parquet",
    "data/test_hmmt_feb_2025/test.parquet"
]
# Must match evaluation/run_inference.sh: 16K responses for Qwen3-1.7B, 8K for Qwen2.5-7B-Instruct.
MODEL="Qwen/Qwen3-1.7B"
OUTPUT_DIR="outputs/inference_results/16384/"


In [ ]:
import os
import json
import datasets
from tqdm.auto import tqdm 
from pathlib import Path

from verl.utils.reward_score import default_compute_score

def load_dataset(parquet_path):
    dataframe = datasets.load_dataset("parquet", data_files=parquet_path)["train"]
    df = dataframe.to_pandas()
    return df

In [ ]:
for parquet_path in TEST_DIRS:
    df = load_dataset(parquet_path)
    original_prompts = df["prompt"].tolist()
    ground_truths = df["reward_model"].apply(lambda x: x["ground_truth"]).values
    data_src = df["data_source"].iloc[0]

    output_filename = f"{Path(data_src).name}_{Path(MODEL).name}_generations.json"
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    with open(output_path, "r") as f:
        generations = json.load(f)

    scores_all = []
    preds_all = []
    for p, gts in tqdm(zip(original_prompts, ground_truths), total=len(original_prompts)):
        po = generations[str(p)]["generations"]
        scores_p = []
        preds_p = []
        for o in po:
            scores_p.append(default_compute_score(data_src, o, gts)==1)
            extracted_pred = extract_solution(o)
            preds_p.append(extracted_pred if extracted_pred else "")
        scores_all.append(scores_p)
        preds_all.append(preds_p)
    print(f"{data_src} pass@1", 100*pass_at_1_from_scores(scores_all))
    print(f"{data_src} cons@32", 100*consensus_at_k(preds_all, k=32, gts=ground_truths))